# Quantifying Wildfire Impacts Using the OPERA DIST-HLS Product

This notebook showcases application of the OPERA Land Surface Disturbance Alert from Harmonized Landsat and Sentinel-2 (`DIST-ALERT-HLS`) product to visualize the impact of a wildfire event that impacted multiple regions of the greater Los Angeles, California area in January 2025.

To explore other use-cases of the OPERA `DIST-ALERT-HLS` product, see our example notebook on [flooding and landslides](https://github.com/OPERA-Cal-Val/OPERA_Applications/blob/main/DIST/DIST_ALERT/Flood_and_Landslide/NorthCarolina_Helene_Flood_Landslides_Sept2024.ipynb).


A [NASA Earthdata](https://earthdata.nasa.gov/) account is required to download the data used in this tutorial. You can create an account [here](https://earthdata.nasa.gov/data/earthdata-login). These credentials must be stored in a `.netrc` file on your local filesystem.

*<font color='red'>Note 1: Please refer to [DIST product specification](https://lpdaac.usgs.gov/documents/1766/OPERA_DIST_HLS_Product_Specification_V1.pdf) for more information. </font>*<br>
*<font color='red'>Note 2: DIST products are distributed via NASA's Distributed Active Archive Centers (DAACs), specifically the [LP DAAC](https://lpdaac.usgs.gov).</font>*

## Background
---
A period of land-use change occurred in a region of Brazil during the 2023 calendar year. In this notebook, we explore the OPERA `DIST-ANN-HLS` product, which captures vegetation disturbance associated with such land-use changes.

### The OPERA `DIST-HLS` Product Suite
---
The OPERA Land Surface Disturbance from Harmonized Landsat and Sentinel-2 (`DIST-HLS`) product suite maps vegetation disturbance using surface reflectance data from Harmonized Landsat and Sentinel-2 (`HLS`). Disturbance is identified when vegetation cover declines or spectral variation falls outside the historical norm for a given `HLS` pixel.

The suite includes two complementary products:

- **`DIST-ALERT-HLS`**: Detects vegetation disturbance at the native `HLS` cadence (every 2–3 days).
- **`DIST-ANN-HLS`**: Summarizes confirmed disturbances from `DIST-ALERT-HLS` over the previous calendar year.

`DIST-ALERT-HLS` is distributed as 19 GeoTIFF layers plus a metadata file, organized in folders corresponding to each input `HLS` tile.  
`DIST-ANN-HLS` includes 21 GeoTIFF layers and a metadata file per tile.

`DIST-HLS` data are projected onto the Military Grid Reference System (MGRS). Each tile spans 109.8 km², consisting of 3,660 x 3,660 pixels at 30-meter resolution, with ~4.9 km of overlap with neighboring tiles.
Detailed descriptions of the raster layers and their properties are available in the [OPERA DIST-HLS Product Specification Document](https://lpdaac.usgs.gov/documents/1766/OPERA_DIST_HLS_Product_Specification_V1.pdf).

### HLS Data 
---
The Harmonized Landsat and Sentinel-2 (`HLS`) dataset provides surface reflectance (SR) data from two satellite sensors:

- **OLI** (Operational Land Imager) on Landsat-8  
- **MSI** (Multi-Spectral Instrument) on Sentinel-2A and 2B

`HLS` data are projected onto the Military Grid Reference System (MGRS). Each tile spans 109.8 km², consisting of 3,660 x 3,660 pixels at 30-meter resolution, with ~4.9 km of overlap with neighboring tiles.

### Accessing the Data
---
`DIST-HLS` products are publicly available via NASA’s Distributed Active Archive Centers (DAACs), specifically through the [LP DAAC](https://lpdaac.usgs.gov).

## Import Libraries and set working directory
Ensure that following dependencies are installed into you environement. This notebook was developed using the `opera_app` environment, which can be found in the [OPERA Applications Github repository](https://github.com/OPERA-Cal-Val/OPERA_Applications).

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import os
from datetime import datetime
import numpy as np
import leafmap
import pandas as pd
import rasterio
import shutil

In [ ]:
# Make and set working directory (modify to your desired path)
working_directory = os.path.expanduser('Greater_LA_fires_Jan2025')
os.makedirs(working_directory, exist_ok=True)

# Change to working directory
os.chdir(working_directory)

# Verify the working directory
print("Current working directory:", os.getcwd())

## Authentication 
A [NASA Earthdata Login](https://urs.earthdata.nasa.gov/) account is required to download the data used in this tutorial. You can create an account at the link provided. After establishing an account, the code in the next cell will verify authentication. If this is your first time running the notebook, you will be prompted to enter your Earthdata login credentials, which will be saved in ~/.netrc.

In [ ]:
leafmap.nasa_data_login()

## View NASA Earthdata datasets
A tab separated values (TSV) file, made available through the opengeos Github repository, catalogues metadata for more than 9,000 datasets available through NASA Earthdata. In the next cell we load the TSV into a pandas dataframe and view the metadata for the first five (5) Earthdata products

In [ ]:
### View Earthdata datasets
earthdata_url = 'https://github.com/opengeos/NASA-Earth-Data/raw/main/nasa_earth_data.tsv'
earthdata_df = pd.read_csv(earthdata_url, sep='\t')
earthdata_df.head()

## View the available OPERA products
Note above that the `earthdata_df` contains a number of columns with metadata about each available product. the `ShortName` column will be used to produce a new dataframe containing only OPERA products. Let's view the available products and their metadata.

In [ ]:
opera_df = earthdata_df[earthdata_df['ShortName'].str.contains('OPERA', case=False)]
opera_df.head()

## Define an area of interest (AOI) and time period of interest (TOI)
Define an area of interest (AOI) for the wildfire event.

In [ ]:
### This cell initializes the AOI and TOI.
AOI = (-120.019, 33.528, -116.219, 34.540) #W, S, E, N; Los Angeles, CA, USA

StartDate_PostFire="2025-1-11T00:00:00"  #Post-fire image start date
EndDate_PostFire="2025-1-14T23:59:59"    #Post-fire image end date

## Query Earthdata and return metadata for OPERA products within the AOI
This notebook utilizes the `leafmap` and `earthaccess` python libraries to facility easier access and vizualization of OPERA data products for a user-specified area of interest (AOI). `leafmap` provides a suite of tools for interactive mapping and visualization in Jupyter Notebooks. `leafmap` version 0.30.0 and and later offer tools specifically for accessing NASA Earthdata by building on the newly developed NASA Earthaccess library. `earthaccess` provides streamlined access to NASA Earthdata and simplifies the authentication and querying to NASA's Common Metadata Repository (CMR) over previously developed approaches.

In the next cell, the user should specify which OPERA product and the date range of interest. The AOI defined previously is used as the boundary in the query.

### View OPERA Product Shortnames

In [ ]:
### Print the available OPERA datasets 
print('Available OPERA datasets:', opera_df['ShortName'].values)

### Query the OPERA DIST-ALERT-HLS dataset for the AOI


In [ ]:
dist_results_PostFire, dist_gdf_PostFire = leafmap.nasa_data_search(
    short_name='OPERA_L3_DIST-ALERT-HLS_V1',
    cloud_hosted=True,
    bounding_box= AOI,
    temporal=(StartDate_PostFire, EndDate_PostFire),
    count=-1,  # use -1 to return all datasets
    return_gdf=True,
)

### See the available DSWx-HLS layers
Functionality within earthaccess enables more more asthetic views of the available layers, as well as displaying the thumbnail. These links are clickable and will download in the browser when clicked. 

In [ ]:
dist_results_PostFire[0] #Note this just shows a single MGRS/HLS tile

### View the DIST-ALERT-HLS metadata and footprints

In [ ]:
### Plot the location of the tiles 
dist_gdf_PostFire.explore(fill=False)

### Extract the `VEG-DIST-STATUS`, `VEG-ANOM-MAX`, and `VEG-DIST-DATE` layers from the dataframe
Downloading all 19 `DIST-ALERT-HLS` layers for all of the intersecting granules will generate a large quantity of local data. Here extract a subset of layers, namely `VEG-DIST-STATUS`, `VEG-ANOM-MAX`, and `VEG-DIST-DATE` for analysis. This can be modified by the end-user if exploration of other `DIST-ALERT-HLS` layers is desired for this event.

In [ ]:
# Define layers of interest
target_layers = ['VEG-DIST-STATUS', 'VEG-ANOM-MAX', 'VEG-DIST-DATE']

# Function to filter granule links based on target_layers
def filter_granule_links(granule, target_substrings):
    original_data_links = granule.data_links

    def filtered_data_links(*args, **kwargs):
        original_links = original_data_links(*args, **kwargs)
        return [
            url for url in original_links
            if url.startswith("https://")
            and url.endswith(".tif")
            and any(sub in url for sub in target_substrings)
        ]

    return filtered_data_links

for granule in dist_results_PostFire:
    granule.data_links = filter_granule_links(granule, target_layers)

## Download data with leafmap
Let's download the data from one of our above queries. In the cell below we specify data from the DIST-ALERT-HLS.

### Create a subdirectory
This will be where the files are downloaded. It will be a subdirectory inside of a directory called `data`, and the directory name will be the date that it was created.

In [ ]:
def create_data_directory():
    current_datetime = datetime.now().strftime("%m_%d_%Y_%H-%M-%S")

    # Define the base directory
    base_directory = "data"

    # Create the full path for the new directory
    new_directory_path_PostFire = os.path.join(base_directory, f"data_{current_datetime}")

    # Create the new directory
    os.makedirs(new_directory_path_PostFire, exist_ok=True)

    print(f"Directory '{new_directory_path_PostFire}' created successfully.")

    return new_directory_path_PostFire

directory_path_PostFire = create_data_directory()

### Download the data
The below will download the data to your newly created subdirectory. Look on your file system for a directory `/data/date/` where `date` is the date the directory was created.

In [ ]:
dist_data_PostFire = leafmap.nasa_data_download(dist_results_PostFire, out_dir=directory_path_PostFire)     

### Filter to post-event disturbance
We would like to view only disturbance detected after the LA wildfires. To do so, we produce a 'filtered' derivative of the DIST-ALERT-HLS layers based on the date of intitial disturbance detection (from the VEG-DIST-DATE layer). Pixel values in the VEG-DIST-DATE layer correspond the date of initial disturbance detection in days since December 31, 2020. Because the wildfire began 7 January 2025, we filter out disturbance that occurred before this date, corresponding to VEG-DIST-DATE pixel values less than 1467 (the number of since 12/31/2020).

Below we calculate the number of days since December 31, 2020 and store it as a variable, which will be used as input in a subsequent step.

In [ ]:
# Define the reference date and the target date
reference_date = datetime(2020, 12, 31)
target_date = datetime(2025, 1, 6)

# Calculate the difference between the two dates
delta = target_date - reference_date

# Get the number of days from the timedelta object
days_since_reference = delta.days

print("Number of days:", days_since_reference)

For this exercise, we are only interested in disturbance detected after the wildfire began on 01/07/2025. Below, we compute a new set of raster layers that are filtered to contain only data following date of interest (day 1467, in this case).

In [ ]:
# Initialize a dictionary to store unique IDs and their corresponding file names
layers_dict = {}

# Iterate over files in the directory
for filename in os.listdir(directory_path_PostFire):
    # Check if the filename matches the expected format
    if filename.endswith('.tif'):
        # Split the filename by underscores
        parts = filename.split('_')
        
        # Ensure there are enough parts to avoid index errors
        if len(parts) >= 5:
            # Extract the unique ID (assuming it’s at index 3)
            unique_id = parts[3]  # Adjust the index if needed
            
            # Add the full filename to the dictionary under the corresponding unique ID
            if unique_id not in layers_dict:
                layers_dict[unique_id] = []  # Initialize an empty list if ID not in dictionary
            
            layers_dict[unique_id].append(filename)

# Print the dictionary of unique IDs and their associated file names
print("Dictionary of unique IDs and corresponding files:")
for unique_id, files in layers_dict.items():
    print(f"{unique_id}: {files}")

In [ ]:
# Function to generate filtered rasters
def generate_filtered_rasters(layers_dict, threshold, output_subdir='filtered'):
    # Ensure the output subdirectory exists
    output_dir = os.path.join(directory_path_PostFire, output_subdir)
    os.makedirs(output_dir, exist_ok=True)

    # Process each unique_id
    for unique_id, file_list in layers_dict.items():
        # Identify the reference _VEG-DIST-DATE.tif file
        date_file = next((f for f in file_list if f.endswith('_VEG-DIST-DATE.tif')), None)
        if not date_file:
            print(f"Warning: No '_VEG-DIST-DATE.tif' file found for {unique_id}")
            continue

        # Open the _VEG-DIST-DATE.tif file and create a mask based on the threshold
        date_file_path = os.path.join(directory_path_PostFire, date_file)
        with rasterio.open(date_file_path) as src:
            date_data = src.read(1)  # Read the first (and only) band
            date_mask = date_data >= threshold  # Mask where date data exceeds the threshold

        # Apply the mask to each layer file
        for file in file_list:
            # If the file is _VEG-DIST-DATE.tif, apply the threshold and save a filtered version
            if file == date_file:
                with rasterio.open(date_file_path) as src:
                    date_filtered_data = np.where(date_mask, date_data, src.nodata)  # Apply the mask
                    date_filtered_filename = file.replace('.tif', '_filtered.tif')  # Update filename
                    date_filtered_path = os.path.join(output_dir, date_filtered_filename)
                    
                    # Save the filtered _VEG-DIST-DATE.tif raster
                    src_meta = src.meta
                    src_meta.update({"nodata": src.nodata})
                    with rasterio.open(date_filtered_path, 'w', **src_meta) as dest:
                        dest.write(date_filtered_data, 1)  # Write to the first band

                print(f"Generated filtered file: {date_filtered_filename}")
                continue  # Move to the next file in the list

            # If file is _DATA-MASK.tif, copy it directly to the output directory with "_filtered" added
            if file.endswith('_DATA-MASK.tif'):
                data_mask_filtered_filename = file.replace('.tif', '_filtered.tif')
                data_mask_filtered_path = os.path.join(output_dir, data_mask_filtered_filename)
                shutil.copy(os.path.join(directory_path_PostFire, file), data_mask_filtered_path)
                
                print(f"Copied _DATA-MASK file: {data_mask_filtered_filename}")
                continue  # Move to the next file in the list

            else:
                print(f"Processing file: {file}")
                # Open the layer file
                file_path = os.path.join(directory_path_PostFire, file)
                with rasterio.open(file_path) as src:
                    layer_data = src.read(1)  # Read the first band
                    layer_meta = src.meta  # Metadata to use for the output file
                    layer_nodata = src.nodata  # Get the 'nan' value for this layer

                    # Apply the mask: where date_mask is False, set layer_data to layer_nodata
                    filtered_data = np.where(date_mask, layer_data, layer_nodata)

                    # Update the filename to include "_filtered"
                    filtered_filename = file.replace('.tif', '_filtered.tif')
                    filtered_file_path = os.path.join(output_dir, filtered_filename)

                    # Save the filtered raster with the same metadata
                    layer_meta.update({"nodata": layer_nodata})
                    with rasterio.open(filtered_file_path, 'w', **layer_meta) as dest:
                        dest.write(filtered_data, 1)  # Write to the first band
                print(f"Generated filtered file: {filtered_filename}")

# Call the function using the 'days_since_reference' variable as the threshold
generate_filtered_rasters(layers_dict, days_since_reference)


## Merge filtered rasters
Below we produce a mosaicked version of each layer, which are saved in a new directory called `merged`

In [ ]:
import os
import rasterio
from rasterio.merge import merge

def merge_filtered_rasters(filtered_rasters_dir):
    
    # Create a directory for merged rasters
    merged_dir = os.path.join(filtered_rasters_dir, 'merged')
    os.makedirs(merged_dir, exist_ok=True)
    
    # Dictionary to hold lists of filenames for each layer
    layer_dict = {}

    # Iterate through filtered rasters to populate the dictionary
    for filename in os.listdir(filtered_rasters_dir):
        if filename.endswith('_filtered.tif'):
            # Split the filename to extract the layer name
            parts = filename.split('_')
            if len(parts) >= 6:  # Check if there are enough parts to avoid index errors
                layer_name = parts[-2]  # Extract the layer name (second-to-last part)
            if layer_name not in layer_dict:
                layer_dict[layer_name] = []
            layer_dict[layer_name].append(os.path.join(filtered_rasters_dir, filename))

    # Merge rasters for each layer and save them
    for layer_name, files in layer_dict.items():
        # Open the rasters and extract nodata values
        src_files_to_mosaic = [rasterio.open(f) for f in files]
        
        # Get the consistent nodata value for the layer
        nodata_value = src_files_to_mosaic[0].nodata
        
        # Perform the merge with preference for data pixels
        mosaic, out_trans = merge(src_files_to_mosaic, nodata=nodata_value, method='first')
        
        # Create metadata for the merged raster
        out_meta = src_files_to_mosaic[0].meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": mosaic.shape[1],
            "width": mosaic.shape[2],
            "transform": out_trans,
            "nodata": nodata_value
        })

        # Save the merged raster
        merged_filename = f"{layer_name}_merged.tif"
        merged_filepath = os.path.join(merged_dir, merged_filename)

        with rasterio.open(merged_filepath, 'w', **out_meta) as dest:
            dest.write(mosaic)

        print(f"Merged raster saved as: {merged_filename}")

        # Close all opened raster files
        for src in src_files_to_mosaic:
            src.close()

    return layer_dict

# Call the function with the filtered_rasters directory
filtered_rasters_dir = os.path.join(directory_path_PostFire, 'filtered')
layer_dict = merge_filtered_rasters(filtered_rasters_dir)

### Add symbology to tiffs
Some of the DIST-ALERT-HLS layers have embedded symbology. Below we add this symbology to the merged raster layers. If they do not have embedded symbology, they are skipped.

In [ ]:
def add_symbology(merged_raster_path, reference_symbology_path):
    try:
        # Read the reference symbology raster
        with rasterio.open(reference_symbology_path) as src:
            # Check if the symbology raster has a colormap
            if 1 in src.colormap(1):
                src_colormap = src.colormap(1)  # Assuming symbology is in band 1
            else:
                print(f"No colormap found for {reference_symbology_path}")
                return  # Exit if no colormap exists

        # Open the merged raster in write mode
        with rasterio.open(merged_raster_path, 'r+') as dst:
            # Write the color map to the first band
            dst.write_colormap(1, src_colormap)

    except Exception as e:
        print(f"Symbology not present for {reference_symbology_path}: {e}...skipping")

symbology_layers = {}

# Loop over each layer in the layer_dict
for layer in layer_dict:
    # Check if we already found a file for this layer
    if layer not in symbology_layers:
        # Loop over each file in the directory
        for filename in os.listdir(directory_path_PostFire):
            # Check if the file is a .tif file
            if filename.endswith('.tif'):
                # Extract the layer name (last part before the extension)
                layer_name = filename.split('_')[-1].split('.')[0]
                
                # Check if the layer name matches the current layer
                if layer_name == layer:
                    # Save the full file path for the first match
                    symbology_layers[layer] = os.path.join(directory_path_PostFire, filename)
                    break  # Stop once the first file for this layer is found

# Print the found file paths for each layer
for layer, path in symbology_layers.items():
    print(f"Layer {layer}: {path}")

# Apply symbology to each merged raster
for layer in layer_dict:
    merged_path = os.path.join(filtered_rasters_dir, 'merged', f"{layer}_merged.tif")
    symbology_path = symbology_layers.get(layer)
    if symbology_path:
        add_symbology(merged_path, symbology_path)
    else:
        print(f"No symbology file found for layer: {layer}")


## View the files using Leafmap
We can visualize a few of the layers using the `leafmap` library. After the map renders, click the upper right corner to toggle on/off the layers.

In [ ]:
import os
import leafmap

# Initialize map
m = leafmap.Map(center=[40, -100], zoom=4)
m.add_basemap("Esri.WorldImagery")

# Define colormap settings for specific layers
colormap_dict = {
    'VEG-ANOM-MAX': 'Reds',
    'VEG-DIST-DATE': 'Blues',
    # No custom colormap for 'VEG-DIST-STATUS', so default will be used
}

# Define layers you want to visualize
layers_viz = ['VEG-DIST-STATUS', 'VEG-ANOM-MAX', 'VEG-DIST-DATE']

# Add rasters to map
for layer in layers_viz:
    merged_path = os.path.join(filtered_rasters_dir, 'merged', f"{layer}_merged.tif")
    if os.path.exists(merged_path):
        cmap = colormap_dict.get(layer, None)
        m.add_raster(merged_path, layer_name=layer, opacity=0.6, colormap=cmap)

# Show layer control
m.add_layer_control()

# Show the map
m

Zoom into the map to view regions of disturbance due to the January 2025 LA fires. Click the widget in the upper right corner of the map to toggle on and off specific layers.

## Layers and Pixel Values

### **Maximum Vegetation Anomaly Value (VEG_ANOM_MAX)**
***

**Data Type:** UInt8<br>
**Description:** Difference between historical and current year observed vegetation cover at the date of maximum decrease, measured on scale from 0-100%<br>

**Layer Values:**<br> 
**0-100:** Maximum loss in % vegetation.<br>

---

### **Date of Initial Vegetation Disturbance (VEG_DIST_DATE)**
***

**Data Type:** Int16<br>
**Description:** Day of first loss anomaly detection in the last year, denoted as the number of days since December 31st, 2020.<br>

**Layer Values:**<br>
**-1:** No data <br>
**0:** No vegetation anomalies in the last year.
**>0:** Day of initial anomaly detection in the last year.

---

### **Vegetation Disturbance Status (VEG_DIST_STATUS)**
***

**Data Type:** UInt8<br>
**Description:** Indication of vegetation cover loss (vegetation disturbance); "provisional" is used from the second detection until vegetation disturbance is detected for consecutive number of HLS scenes, when it is then labeled "confirmed."<br>

**Layer Values:**<br> 
* **0:** No disturbance<br>
* **1:** First disturbance detection with vegetation cover change <50% <br>
* **2:** Provisional (**two consecutive disturbance detections**) with vegetation cover change <50% <br>
* **3:** Confirmed (**recurrent detection**) Disturbance with vegetation cover change <50% <br> 
* **4:** First disturbance detection with vegetation cover change ≥50% <br>
* **5:** Provisional (**two consecutive disturbance detections**) with vegetation cover change ≥50% <br>
* **6:** Confirmed (**recurrent detection**) Disturbance with vegetation cover change ≥50% <br> 
* **7:** Confirmed (**recurrent detection**) Disturbance with vegetation cover change <50%, completed <br> 
* **8:** Confirmed (**recurrent detection**) Disturbance with vegetation cover change ≥50%, completed <br> 
* **255:** No data

*<font color='red'>For more detail about the layers not visualized here, please see the [DIST product specification](https://lpdaac.usgs.gov/documents/1766/OPERA_DIST_HLS_Product_Specification_V1.pdf). </font>*<br>